# Statistical Properties of Financial Data — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**


In [ ]:
!pip install yfinance seaborn statsmodels --quiet
import yfinance as yf
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import statsmodels.graphics.tsaplots as tsaplots
import warnings; warnings.filterwarnings('ignore')

plt.rcParams.update({'figure.facecolor':'white','axes.facecolor':'white',
    'axes.spines.top':False,'axes.spines.right':False,'axes.grid':True,'grid.alpha':0.3})
YELLOW='#FDE70E'; ORANGE='#FCB310'; RED='#C70101'; GREY='#4B4B4B'

# Shared download
START = '2019-01-01'; END = '2024-12-31'
tickers_all = ['AAPL','MSFT','JPM','XOM','SPY','TLT','GLD','BTC-USD','^GSPC']
raw = yf.download(tickers_all, start=START, end=END, auto_adjust=True, progress=False)
all_prices = raw['Close'].copy()
all_prices = all_prices.rename(columns={'^GSPC':'SP500'})
all_ret    = all_prices.pct_change().dropna(how='all') * 100
print('✓ Data loaded.')

---
# Solution 1 — Match the Statistic

| # | Question | Statistic |
|---|----------|-----------|
| a | Typical daily move | **Mean** |
| b | Spread of returns | **Std deviation** |
| c | Losses vs. gains asymmetry | **Skewness** |
| d | Frequency of extremes vs. Normal | **Excess kurtosis** |
| e | Is the distribution Normal? | **Jarque-Bera p-value** |
| f | Worst day | **Min** |

---
# Solution 2 — Descriptive Statistics for a Swiss Stock

In [ ]:
TICKER_EX = 'NESN.SW'
data_ex   = yf.download(TICKER_EX, start=START, end=END, auto_adjust=True, progress=False)
prices_ex = data_ex['Close'].squeeze()
ret_ex    = prices_ex.pct_change().dropna() * 100

print(f'Stock: {TICKER_EX}   ({len(prices_ex)} days)')
print('\nDescriptive statistics (daily returns, %):')
print(ret_ex.describe().round(4))
print(f'\nSkewness:        {ret_ex.skew():.3f}')
print(f'Excess kurtosis: {ret_ex.kurtosis():.3f}')
print(f'\nAnnualised mean: {ret_ex.mean()*252:.2f}%')
print(f'Annualised vol:  {ret_ex.std()*np.sqrt(252):.2f}%')

**Answers:**
1. NESN.SW daily mean ≈ 0.02%, annualised ≈ 5%. (A modest positive drift typical of a defensive blue-chip.)
2. Daily vol ≈ 0.95%, annualised ≈ 15% — clearly below the S&P 500 (~20%) and well below tech names. Consistent with Nestlé's defensive, consumer-staples profile.
3. Skewness is slightly negative (≈ −0.3 to −0.5): large losses are somewhat more frequent than equally large gains.
4. Excess kurtosis is well above 0 (typically 5–10): extreme moves occur far more often than a Normal would predict — the classic 'fat tails' pattern.


---
# Solution 3 — Annualisation by Hand

In [ ]:
# Scenario A: S&P 500-like
daily_mean_A, daily_std_A = 0.04, 1.1
ann_mean_A = daily_mean_A * 252
ann_vol_A  = daily_std_A  * np.sqrt(252)
print(f'A: ann_mean = {ann_mean_A:.2f}%   ann_vol = {ann_vol_A:.2f}%')

# Scenario B: TSLA-like
daily_mean_B, daily_std_B = 0.10, 2.5
ann_mean_B = daily_mean_B * 252
ann_vol_B  = daily_std_B  * np.sqrt(252)
print(f'B: ann_mean = {ann_mean_B:.2f}%   ann_vol = {ann_vol_B:.2f}%')

# Scenario C: monthly std = 4%
monthly_std_C = 4.0
ann_vol_C = monthly_std_C * np.sqrt(12)
print(f'C: ann_vol  = {ann_vol_C:.2f}%   (4% × √12 ≈ 4 × 3.46)')

**Answers:**
1. Variance scales linearly with the number of independent periods. For 252 daily observations, annual variance = 252·σ²_daily, so annual std = σ_daily·√252. For 12 monthly obs, annual std = σ_monthly·√12.
2. Daily vol ≈ 32% / √252 ≈ 32% / 15.87 ≈ 2.0%.
3. Valid under i.i.d. returns with finite variance and no autocorrelation. It breaks down when (i) volatility is time-varying / clustered, (ii) returns have heavy autocorrelation, or (iii) horizons are very long, where mean drift starts dominating.


---
# Solution 4 — Skewness & Kurtosis Across Assets

In [ ]:
tickers_ex4 = ['SPY','TLT','GLD','BTC-USD']
ret4 = all_ret[tickers_ex4].dropna()

summary4 = pd.DataFrame({
    'Mean':     ret4.mean().round(4),
    'Std':      ret4.std().round(4),
    'Skewness': ret4.skew().round(3),
    'ExKurt':   ret4.kurtosis().round(3),
    'Min':      ret4.min().round(2),
    'Max':      ret4.max().round(2),
})
print('Higher Moments — Daily Returns 2019–2024 (%):')
print(summary4)

**Answers:**
1. SPY typically has the most negative skewness — equity drawdowns tend to be sharper than rallies. Strong negative skew = elevated crash risk relative to symmetric models.
2. Excess kurtosis is highest for SPY or BTC-USD (both can exceed 10). BTC-USD adds 24/7 trading and reflexivity; SPY shows the classic equity fat tails. Matches intuition: equities and crypto routinely produce 5σ days that a Normal would call once-in-a-lifetime.
3. SPY or BTC-USD — both have heavy tails and (for SPY) negative skew, so a Normal would systematically underestimate VaR and option tail prices.


---
# Solution 5 — Jarque-Bera Test

In [ ]:
print(f'{"Asset":<10} | {"Skew":>7} | {"ExKurt":>7} | {"JB":>10} | {"p-value":>10} | Decision')
print('-'*70)
for col in ret4.columns:
    r = ret4[col].dropna()
    jb_stat, jb_p = stats.jarque_bera(r)
    decision = 'Reject H₀' if jb_p < 0.05 else 'Fail to reject'
    print(f'{col:<10} | {r.skew():>7.3f} | {r.kurtosis():>7.3f} | {jb_stat:>10.1f} | {jb_p:>10.2e} | {decision}')

**Answers:**
1. We reject H₀ for every asset — SPY, TLT, GLD, and BTC-USD all have p-values far below 0.05. This is the rule, not the exception, for daily financial data.
2. p ≈ 1e-50 means: if returns were truly Normal, you'd see this much skewness/kurtosis with probability essentially zero. Practically, treat the data as decisively non-Normal.
3. A VaR model assuming Normal will systematically understate the probability of large losses — your '99% one-day VaR' will be breached far more than 1% of the time. Risk capital is then under-reserved.


---
# Solution 6 — Histogram vs. Normal

In [ ]:
asset = 'AAPL'
r = all_ret[asset].dropna()
mu, sd = r.mean(), r.std()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
x = np.linspace(r.min(), r.max(), 500)
for ax, scale in zip(axes, ['linear','log']):
    ax.hist(r, bins=80, density=True, color=GREY, alpha=0.7, label=asset)
    ax.plot(x, stats.norm.pdf(x, mu, sd), color=RED, lw=2, label='Normal(μ,σ)')
    if scale == 'log':
        ax.set_yscale('log')
        ax.set_title('Log scale — fat tails visible', fontweight='bold')
    else:
        ax.set_title('Linear scale — peak too tall', fontweight='bold')
    ax.set_xlabel('Daily return (%)'); ax.set_ylabel('Density')
    ax.legend()
plt.tight_layout(); plt.show()

worst = r.min(); n_sigma = (worst - mu)/sd
p_normal = stats.norm.cdf(n_sigma)
freq_days = 1/(2*p_normal) if p_normal > 0 else float('inf')
print(f'Worst day: {worst:.2f}%  →  {n_sigma:.2f} σ from mean')
print(f'Under Normal: ~1 in {freq_days:,.0f} days  (sample only has {len(r)} days)')

**Answers:**
1. The empirical histogram is too tall and narrow at the centre — many more days cluster near zero than a Normal predicts.
2. On the log scale, the empirical density stays well above the Normal in both tails — that is the visual signature of fat tails.
3. AAPL's worst day (e.g. ~−13% in March 2020) sits roughly 6–8σ below the mean.
4. Under Normal, an 8σ event has probability ≈ 1e-15 — once in trillions of days. It happened in our 6-year sample. The Normal model is not just slightly wrong here; it is wrong by many orders of magnitude.


---
# Solution 7 — Q-Q Plot

In [ ]:
asset = 'AAPL'
r = all_ret[asset].dropna().values

np.random.seed(42)
sim = np.random.normal(r.mean(), r.std(), len(r))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
for ax, data, title in [(axes[0], r,   f'{asset} — Q-Q vs. Normal'),
                         (axes[1], sim, 'Simulated Normal — reference')]:
    stats.probplot(data, dist='norm', plot=ax)
    ax.set_title(title, fontweight='bold')
    ax.get_lines()[0].set_markerfacecolor(GREY)
    ax.get_lines()[0].set_markeredgecolor(GREY)
    ax.get_lines()[0].set_markersize(3)
    ax.get_lines()[1].set_color(RED)
    ax.get_lines()[1].set_linewidth(2)
plt.tight_layout(); plt.show()

**Answers:**
1. Deviation is largest at both extremes: bottom-left points fall *below* the line; top-right points sit *above* it.
2. That S-shape (or 'reverse-S' depending on orientation) is the classic fat-tailed signature: extreme observations exceed what Normal quantiles predict on both sides.
3. The simulated Normal hugs the red line tightly across the whole range — the contrast makes the empirical deviation unmistakable.


---
# Solution 8 — Volatility Clustering

In [ ]:
asset = 'AAPL'
p = all_prices[asset].dropna()
r = all_ret[asset].dropna()

fig, axes = plt.subplots(2, 1, figsize=(13, 7), sharex=True,
                          gridspec_kw={'height_ratios':[2,1], 'hspace':0.05})
axes[0].plot(p.index, p.values, color='black', lw=1.2)
axes[0].fill_between(p.index, p.values, alpha=0.06, color='black')
axes[0].set_ylabel('Price (USD)')
axes[0].set_title(f'{asset} — Price & Daily Returns', fontweight='bold')

axes[1].bar(r.index, r.values,
            color=[RED if v<0 else 'black' for v in r.values], width=1.0)
axes[1].axhline(0, color=GREY, lw=0.8)
axes[1].set_ylabel('Daily return (%)'); axes[1].set_xlabel('Date')

axes[1].axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-30'),
                color=ORANGE, alpha=0.15, label='COVID crash')
axes[1].axvspan(pd.Timestamp('2022-04-01'), pd.Timestamp('2022-12-31'),
                color=YELLOW, alpha=0.20, label='2022 bear market')
axes[1].legend(loc='lower left')
plt.tight_layout(); plt.show()

**Answers:**
1. Two clear clusters of large bars: Feb–Apr 2020 (COVID) and most of 2022 (rate-hike bear market).
2. Mid-2021 and early 2024 are notably calm — small bars dominate for weeks.
3. Under i.i.d. returns, the size of today's move would carry no information about tomorrow's. Empirically, big moves cluster, so |r_t| is correlated with |r_{t+1}| — directly contradicting i.i.d.


---
# Solution 9 — ACF: Returns vs. Squared Returns

In [ ]:
asset = 'AAPL'
r = all_ret[asset].dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
tsaplots.plot_acf(r,    lags=30, ax=axes[0], color='black', vlines_kwargs={'colors':'black'})
axes[0].set_title('ACF of Returns — ≈ 0', fontweight='bold')
axes[0].set_ylim(-0.15, 0.4)

tsaplots.plot_acf(r**2, lags=30, ax=axes[1], color=RED, vlines_kwargs={'colors':RED})
axes[1].set_title('ACF of Squared Returns — high & persistent', fontweight='bold')
axes[1].set_ylim(-0.15, 0.4)
plt.tight_layout(); plt.show()

**Answers:**
1. Yes — almost all return ACF spikes lie inside the blue 95% confidence band. Returns themselves have essentially no linear predictability.
2. Squared-return spikes sit clearly *outside* the band, often for many lags, decaying slowly.
3. The combination — unpredictable signs but predictable magnitudes — is the empirical motivation for **GARCH** models (and the broader ARCH family).
4. *Direction is unpredictable, risk is predictable.* Markets are roughly efficient in mean but not in variance.


---
# Solution 10 — Crisis vs. Normal Correlations

In [ ]:
tickers_ex10 = ['AAPL','MSFT','JPM','XOM','SP500']
panel = all_ret[tickers_ex10].dropna()

crisis = panel.loc['2020-02-01':'2020-05-31']
normal = panel.loc['2021-01-01':'2021-12-31']
corr_c = crisis.corr()
corr_n = normal.corr()

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, mat, title in [(axes[0], corr_n, f'Normal: 2021 ({len(normal)} days)'),
                        (axes[1], corr_c, f'Crisis: Feb–May 2020 ({len(crisis)} days)')]:
    sns.heatmap(mat, annot=True, fmt='.2f', cmap='RdYlGn_r', center=0,
                vmin=-1, vmax=1, square=True, linewidths=0.5, ax=ax)
    ax.set_title(title, fontweight='bold')
plt.tight_layout(); plt.show()

def avg_off(c):
    m = c.values; n = m.shape[0]
    return (m.sum() - n) / (n*(n-1))

diff = corr_c - corr_n
print(f'Average off-diagonal correlation:')
print(f'  Normal:  {avg_off(corr_n):.3f}')
print(f'  Crisis:  {avg_off(corr_c):.3f}')
print(f'  Δ:       {avg_off(corr_c) - avg_off(corr_n):+.3f}')

# Largest jump
off = diff.where(np.triu(np.ones_like(diff, dtype=bool), k=1))
i, j = np.unravel_index(np.nanargmax(off.values), off.shape)
print(f'\nLargest pairwise jump: {diff.index[i]} ↔ {diff.columns[j]}  '
      f'({corr_n.iloc[i,j]:.2f} → {corr_c.iloc[i,j]:.2f})')

**Answers:**
1. Average correlation rises by roughly +0.20–0.30 from 2021 to the COVID crash window.
2. The largest pairwise jump is typically between sectors that normally diversify each other (e.g. JPM ↔ XOM or AAPL ↔ XOM) — they all sell off together in panic.
3. Diversification is a function of correlation. When correlations spike to 0.9+ in a crisis, a 'diversified' portfolio behaves like a single asset — exactly when you need protection most.
4. **Correlation breakdown** (or 'diversification breakdown' / 'flight-to-correlation').


---
# 🔥 Challenge A — Rolling Correlation

In [ ]:
WINDOW = 60
pair = ['AAPL','SP500']
panel = all_ret[pair].dropna()

roll_corr = panel['AAPL'].rolling(WINDOW).corr(panel['SP500'])

fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(roll_corr.index, roll_corr.values, color=RED, lw=1.4)
ax.fill_between(roll_corr.index, roll_corr.values, alpha=0.15, color=RED)
ax.axhline(roll_corr.mean(), color=GREY, ls='--', lw=1,
           label=f'Average: {roll_corr.mean():.2f}')
ax.axvspan(pd.Timestamp('2020-02-20'), pd.Timestamp('2020-04-30'),
           color=ORANGE, alpha=0.15, label='COVID crash')
ax.set_ylabel(f'{WINDOW}-day rolling correlation')
ax.set_title('Rolling Correlation: AAPL vs. S&P 500', fontweight='bold')
ax.set_ylim(0, 1); ax.legend()
plt.tight_layout(); plt.show()

print(f'Min:    {roll_corr.min():.2f}  on {roll_corr.idxmin().date()}')
print(f'Max:    {roll_corr.max():.2f}  on {roll_corr.idxmax().date()}')
print(f'Mean:   {roll_corr.mean():.2f}')
print('→ Correlation visibly spikes during the COVID window.')

---
# 🔥 Challenge B — Stylized Facts: Bitcoin vs. S&P 500

In [ ]:
btc = all_ret['BTC-USD'].dropna()
sp  = all_ret['SP500'].dropna()

# 1) Higher moments + JB
rows = []
for name, r in [('BTC-USD', btc), ('SP500', sp)]:
    jb, p = stats.jarque_bera(r)
    rows.append({'Asset': name,
                 'Mean':     round(r.mean(), 4),
                 'Std':      round(r.std(), 4),
                 'Skew':     round(r.skew(), 3),
                 'ExKurt':   round(r.kurtosis(), 3),
                 'JB':       round(jb, 1),
                 'JB p':     f'{p:.2e}',
                 'Ann Vol %': round(r.std()*np.sqrt(252), 2)})
print(pd.DataFrame(rows).to_string(index=False))

# 2) ACF of squared returns
fig, axes = plt.subplots(1, 2, figsize=(14, 4.5))
tsaplots.plot_acf(btc**2, lags=30, ax=axes[0], color=RED, vlines_kwargs={'colors':RED})
axes[0].set_title('BTC-USD — ACF of r²', fontweight='bold'); axes[0].set_ylim(-0.1, 0.4)
tsaplots.plot_acf(sp**2,  lags=30, ax=axes[1], color='black', vlines_kwargs={'colors':'black'})
axes[1].set_title('SP500 — ACF of r²', fontweight='bold');   axes[1].set_ylim(-0.1, 0.4)
plt.tight_layout(); plt.show()

# 3) Rolling vol
fig, ax = plt.subplots(figsize=(13, 4.5))
ax.plot(btc.rolling(21).std()*np.sqrt(252), color=ORANGE, lw=1.3, label='BTC-USD')
ax.plot(sp .rolling(21).std()*np.sqrt(252), color='black',  lw=1.3, label='S&P 500')
ax.set_title('Rolling 21-day annualised volatility', fontweight='bold')
ax.set_ylabel('Annualised vol (%)'); ax.legend()
plt.tight_layout(); plt.show()

print('\nVerdict: BTC shows the SAME stylized facts as equities — fat tails, JB rejected,')
print('persistent ACF of r², clustered volatility — but at roughly 3–4× the magnitude.')

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*